# Response ZIP Code Map

Maps survey response ZIP codes (and the ZIPs adjacent to them) on a full Massachusetts ZIP code map.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

SHP = Path("ZIP Codes File") / "ZIPCODES_NT_POLY.shp"
OUT = Path("response_zip_map.png")

## Response ZIP codes

Paste the list here, comma-separated. Duplicates are fine.

In [ ]:
RESPONSE_ZIPS = """
02186, 02149, 02148, 02149, 02149, 02149, 02149, 02149, 01731, 01731, 02149, 02148,
02150, 02108, 02149, 02149, 02149, 02149, 02149, 02149, 02148, 02421, 01022, 01002,
01022, 10011, 01002, 01062, 01105, 01104, 01104, 01536, 01653, 01752, 01607, 01602,
01527, 01527, 01501, 01527, 01501, 01609, 01527, 01604, 01603, 01854, 01583, 01606,
01590, 01527, 01609, 01527, 01107, 01109, 02125, 01027, 01027
"""
response = {z.strip() for z in RESPONSE_ZIPS.split(",") if z.strip()}
len(response)

## Load ZIPs and find adjacent ones

In [ ]:
# Some ZIPs are split across several polygons; merge them into one shape per ZIP.
zips = gpd.read_file(SHP).dissolve(by="POSTCODE", as_index=False)

missing = sorted(response - set(zips["POSTCODE"]))
if missing:
    print(f"Not in the MA ZIP shapefile (skipped): {', '.join(missing)}")

selected = zips[zips["POSTCODE"].isin(response)]
# Adjacent = shares a boundary with any response ZIP. A small buffer absorbs
# slivers/gaps between polygon edges in the source data.
touch_zone = selected.geometry.buffer(5).union_all()
adjacent = zips[zips.intersects(touch_zone) & ~zips["POSTCODE"].isin(response)]

print(f"Response ZIPs mapped: {len(selected)}  |  Adjacent ZIPs: {len(adjacent)}")
print("Response:", ", ".join(sorted(selected["POSTCODE"])))
print("Adjacent:", ", ".join(sorted(adjacent["POSTCODE"])))

## Map

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
zips.plot(ax=ax, color="#d9d9d9", edgecolor="white", linewidth=0.3)
adjacent.plot(ax=ax, color="#cfe8f7", edgecolor="white", linewidth=0.3)
selected.plot(ax=ax, color="#7cc4ec", edgecolor="white", linewidth=0.3)

ax.legend(
    handles=[
        Patch(color="#7cc4ec", label="Response ZIP code"),
        Patch(color="#cfe8f7", label="Adjacent ZIP code"),
        Patch(color="#d9d9d9", label="Other ZIP codes"),
    ],
    loc="lower left",
    frameon=False,
    fontsize=11,
)
ax.set_title("Survey Response ZIP Codes and Adjacent ZIP Codes, Massachusetts", fontsize=15)
ax.set_axis_off()
fig.tight_layout()
fig.savefig(OUT, dpi=200)
plt.show()

## Interactive web map

Writes `index.html`, a standalone map you can open in a browser or publish with GitHub Pages. Hover over a ZIP code to see its number and town.

In [ ]:
import folium

HTML_OUT = Path("index.html")

COLORS = {"Response ZIP code": "#7cc4ec", "Adjacent ZIP code": "#cfe8f7", "Other ZIP code": "#d9d9d9"}

web = zips[["POSTCODE", "CITY_TOWN", "geometry"]].copy()
web["status"] = "Other ZIP code"
web.loc[web["POSTCODE"].isin(adjacent["POSTCODE"]), "status"] = "Adjacent ZIP code"
web.loc[web["POSTCODE"].isin(response), "status"] = "Response ZIP code"
# Simplify (meters, in the source projection) to keep the HTML file small, then convert to lat/lon.
web["geometry"] = web.geometry.simplify(25)
web = web.to_crs(epsg=4326)

m = folium.Map(location=[42.25, -71.8], zoom_start=8, tiles="CartoDB positron")
folium.GeoJson(
    web,
    style_function=lambda f: {
        "fillColor": COLORS[f["properties"]["status"]],
        "fillOpacity": 0.85,
        "color": "white",
        "weight": 0.6,
    },
    highlight_function=lambda f: {"weight": 2, "color": "#333333"},
    tooltip=folium.GeoJsonTooltip(
        fields=["POSTCODE", "CITY_TOWN", "status"],
        aliases=["ZIP code", "Town", "Category"],
    ),
).add_to(m)

legend_rows = "".join(
    f'<div><span style="display:inline-block;width:14px;height:14px;background:{c};'
    f'margin-right:6px;vertical-align:middle;border:1px solid #999"></span>{label}</div>'
    for label, c in COLORS.items()
)
legend = (
    '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
    'padding:10px 12px;border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);'
    'font:13px/1.6 sans-serif">'
    "<b>Survey Response ZIP Codes</b>" + legend_rows + "</div>"
)
m.get_root().html.add_child(folium.Element(legend))

m.save(HTML_OUT)
print(f"Saved {HTML_OUT} ({HTML_OUT.stat().st_size / 1e6:.1f} MB)")
m